In [1]:
!pip install -U "transformers>=4.44" "datasets>=2.19" "accelerate>=0.34" peft bitsandbytes trl
#!pip install transformers datasets accelerate peft bitsandbytes trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.5/511.5 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14

In [ ]:
# 0) ambiente y seeds
import os, random, math, torch, pandas as pd
SEED = 123
random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

PyTorch: 2.8.0+cu126 | device: cuda


In [ ]:
# ============================================================
# 1) Carga tokenizer + modelo Phi-3.5-mini-instruct en 4-bit
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Modelo base
MODEL_ID = "microsoft/phi-3.5-mini-instruct"

# Verificación para usar bf16 si la GPU lo soporta (Compute Capability >= 8, ej. Ada / Ampere)
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

# Configuración de cuantización (igual que Qwen3)
quant_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16,
)

print("[1] Cargando tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    trust_remote_code=True
)

# Carga el modelo en 4-bit con asignación automática a GPU/CPU según disponibilidad
print("[1] Cargando modelo (4-bit)…")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quant_4bit,
    trust_remote_code=True
)

# Algunos modelos de Phi no traen pad_token definido, lo igualamos al eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# El modelo Phi-3.5-mini tiene un contexto máximo de 16K
MODEL_MAX = getattr(model.config, "max_position_embeddings", 16384)
tokenizer.model_max_length = MODEL_MAX

# Para entrenamiento ligero, reducir longitud de secuencia
MAX_SEQ_LEN_TRAIN = 2048
print("[1] MODEL_MAX:", MODEL_MAX, "| MAX_SEQ_LEN_TRAIN:", MAX_SEQ_LEN_TRAIN)


[1] Cargando tokenizer…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[1] Cargando modelo (4-bit)…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[1] MODEL_MAX: 131072 | MAX_SEQ_LEN_TRAIN: 2048


In [ ]:
# ============================================================
# 2) Preparar k-bit + gradient checkpointing + grads en input
# ============================================================

from peft import prepare_model_for_kbit_training

print("[2] Preparando modelo para k-bit training…")

# Ajusta la preparación para entrenamiento en 4-bit (PEFT)
# Esto coloca los pesos en modo no entrenable, prepara norm layers y dtype correcto
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Activar gradient checkpointing para reducir memoria VRAM
model.gradient_checkpointing_enable()

# Desactivar la caché de atención (necesario para entrenamiento)
model.config.use_cache = False

# Algunos modelos requieren habilitar gradientes en las entradas
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def make_inputs_require_grad(module, input, output):
        output.requires_grad_(True)
    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

print("[2] Listo para fine-tuning en 4-bit ✅")


[2] Preparando modelo para k-bit training…
[2] Listo para fine-tuning en 4-bit ✅


In [ ]:
# ============================================================
# 3) Detectar módulos de atención y aplicar LoRA (Phi-3.5)
# ============================================================

from peft import LoraConfig, get_peft_model

# Módulos más comunes en la arquitectura Phi-3.5 (Tiny-Transformer optimizado)
# Basado en inspección de model.named_modules()
default_targets = [
    "q_proj", "k_proj", "v_proj", "o_proj",           # comunes en modelos LLaMA/Qwen
    "Wqkv", "out_proj",                              # usados en Phi 2/3
    "linear", "mlp.fc1", "mlp.fc2"                   # fallback (algunas versiones)
]

# Autodetección
all_module_names = [n for n, _ in model.named_modules()]
selected = [t for t in default_targets if any(n.endswith(t) for n in all_module_names)]

# Fallback adicional para Phi (casos donde hay 'mixer.Wqkv' y 'mixer.out_proj')
if not selected:
    phi_targets = [n for n in all_module_names if any(k in n for k in ["Wqkv", "out_proj"])]
    if phi_targets:
        selected = ["Wqkv", "out_proj"]

if not selected:
    raise ValueError("[3] No encontré módulos de atención típicos. Ejemplo de nombres:",
                     all_module_names[:40])

print("[3] target_modules LoRA:", selected)

# Configuración LoRA optimizada para Phi-3.5-mini (4-bit)
lora_cfg = LoraConfig(
    r=8,                    # rank: equilibrio entre capacidad y memoria
    lora_alpha=16,          # escala (16–32 funciona bien)
    lora_dropout=0.05,      # regularización ligera
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=selected
)

# Inyectar adaptadores LoRA
model = get_peft_model(model, lora_cfg)

# Verificar porcentaje de parámetros entrenables
def count_trainable_params(m):
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m.parameters())
    return tr, tot, 100 * tr / tot

t, T, pct = count_trainable_params(model)
print(f"[3] Parámetros entrenables: {t:,} / {T:,} ({pct:.4f}%)")


[3] target_modules LoRA: ['v_proj', 'o_proj']
[3] Parámetros entrenables: 1,572,864 / 2,010,713,088 (0.0782%)


In [ ]:
# ============================================================
# 4) Cargar datos (CSV) y vistazo rápido
# ============================================================

import pandas as pd
from pathlib import Path

# Carpeta base con los CSV preprocesados
DATA_DIR = Path("/content/drive/MyDrive/MAIA-PROYECTO")

# Archivos individuales
train_path = DATA_DIR / "data_finetuning_train.csv"
val_path   = DATA_DIR / "data_finetuning_val.csv"
test_path  = DATA_DIR / "data_finetuning_test.csv"

# Leer CSV
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

# Información general
print(f"Train set: {len(train_df):,} muestras")
print(f"Validation set: {len(val_df):,} muestras")
print(f"Test set: {len(test_df):,} muestras")

print("Columnas disponibles:", train_df.columns.tolist())

# Vista rápida de las primeras filas
display(train_df.head(2))


Train set: 3,038 muestras
Validation set: 380 muestras
Test set: 380 muestras
Columnas disponibles: ['name', 'article', 'summary']


,name,article,summary
0,10.1002-14651858.CD010557.pub2,Background\nAlthough antidepressants are often...,Are there effective medications for treating d...
1,10.1002-14651858.CD000938.pub2,Background\nWomen with a suspected large‐for‐d...,Induction of labour at or near the end of preg...


In [ ]:
# ============================================================
# 5) Tokenización — versión optimizada para T4 en Colab
# ============================================================

from datasets import Dataset
import pandas as pd

# Configuraciones clave
USE_CHUNKING = True       # True → divide artículos largos (recomendado)
MAX_TARGET_TOKENS = 384   # resumen máximo
OVERLAP = 64              # menor solapamiento para ahorrar VRAM
MAX_SEQ_LEN_TRAIN = 2048  # ya definido antes

# Prompt base (mantén el mismo estilo del fine-tuning previo)
SYS_PROMPT = (
    "You are a helpful medical/health writer who rewrites complex biomedical text "
    "into Plain Language Summaries understandable for laypeople. "
    "Use short sentences, everyday words, and neutral tone. "
    "Avoid jargon; when unavoidable, define it simply."
)

# ------------------------------------------------------------
# CHUNKING (recomendado para textos >2 000 tokens)
# ------------------------------------------------------------
if USE_CHUNKING:
    PROMPT_PREFIX = SYS_PROMPT + "\n\nScientific Text:\n"
    PROMPT_SUFFIX = "\n\nPlain Language Summary:"

    def tokenize_chunks(article: str, summary: str):
        # tokens del target (resumen)
        tgt_ids = tokenizer(
            summary + tokenizer.eos_token,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_TARGET_TOKENS
        ).input_ids

        prefix_ids = tokenizer(PROMPT_PREFIX, add_special_tokens=False).input_ids
        suffix_ids = tokenizer(PROMPT_SUFFIX, add_special_tokens=False).input_ids

        avail_for_chunk = MAX_SEQ_LEN_TRAIN - len(tgt_ids) - len(prefix_ids) - len(suffix_ids)
        if avail_for_chunk <= 0:
            ids = tgt_ids[-MAX_SEQ_LEN_TRAIN:]
            return [{"input_ids": ids, "labels": ids[:]}]

        art_ids = tokenizer(article, add_special_tokens=False).input_ids
        stride = max(1, avail_for_chunk - OVERLAP)
        examples = []

        for start in range(0, len(art_ids), stride):
            chunk = art_ids[start:start + avail_for_chunk]
            if not chunk:
                break
            inp_ids = prefix_ids + chunk + suffix_ids
            ids = inp_ids + tgt_ids
            labels = [-100] * len(inp_ids) + tgt_ids
            examples.append({"input_ids": ids, "labels": labels})
        return examples

    def df_to_dataset(frame: pd.DataFrame):
        input_ids_list, labels_list = [], []
        for a, s in zip(frame["article"].tolist(), frame["summary"].tolist()):
            for ex in tokenize_chunks(a, s):
                input_ids_list.append(ex["input_ids"])
                labels_list.append(ex["labels"])
        ds = Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})
        ds.set_format(type="torch", columns=["input_ids", "labels"])
        return ds

# ------------------------------------------------------------
# SIN CHUNK (solo truncado)
# ------------------------------------------------------------
else:
    def build_prompt(article: str) -> str:
        return f"{SYS_PROMPT}\n\nScientific Text:\n{article}\n\nPlain Language Summary:"

    def tokenize_one(article: str, summary: str):
        tgt = tokenizer(
            summary + tokenizer.eos_token,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_TARGET_TOKENS
        ).input_ids
        avail = MAX_SEQ_LEN_TRAIN - len(tgt)
        if avail <= 0:
            ids = tgt[-MAX_SEQ_LEN_TRAIN:]
            return {"input_ids": ids, "labels": ids[:]}
        inp = tokenizer(
            build_prompt(article),
            add_special_tokens=False,
            truncation=True,
            max_length=avail
        ).input_ids
        ids = inp + tgt
        labels = [-100] * len(inp) + tgt
        return {"input_ids": ids, "labels": labels}

    def df_to_dataset(frame: pd.DataFrame):
        input_ids_list, labels_list = [], []
        for a, s in zip(frame["article"].tolist(), frame["summary"].tolist()):
            ex = tokenize_one(a, s)
            input_ids_list.append(ex["input_ids"])
            labels_list.append(ex["labels"])
        ds = Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})
        ds.set_format(type="torch", columns=["input_ids", "labels"])
        return ds

print(f"[5] Tokenizador listo. USE_CHUNKING = {USE_CHUNKING}")


[5] Tokenizador listo. USE_CHUNKING = True


In [ ]:
# ============================================================
# 6) Split y construcción de datasets tokenizados
# ============================================================

from datasets import DatasetDict

def safe_df_to_dataset(frame):
    """Convierte un DataFrame en Dataset HF, ignorando filas vacías."""
    frame = frame.dropna(subset=["article", "summary"])
    ds = df_to_dataset(frame)
    ds.set_format(type="torch", columns=["input_ids", "labels"])
    return ds

print("[6] Tokenizando train…")
tok_train = safe_df_to_dataset(train_df)

print("[6] Tokenizando validation…")
tok_val = safe_df_to_dataset(val_df)

print("[6] Tokenizando test…")
tok_test = safe_df_to_dataset(test_df)

# DatasetDict para usar con Trainer
tok = DatasetDict({
    "train": tok_train,
    "validation": tok_val,
    "test": tok_test
})

print(tok)
print(f"[6] Tokens por split: train={len(tok_train)}, val={len(tok_val)}, test={len(tok_test)}")
print("[6] Ejemplo:", tok_train[0])


[6] Tokenizando train…
[6] Tokenizando validation…
[6] Tokenizando test…
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3914
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 486
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 489
    })
})
[6] Tokens por split: train=3914, val=486, test=489
[6] Ejemplo: {'input_ids': tensor([  887,   526,   263,  ...,   573, 25828,  4835]), 'labels': tensor([ -100,  -100,  -100,  ...,   573, 25828,  4835])}


In [ ]:
# ============================================================
# 7) collator + sanity forward/backward (Phi-3.5-mini, T4)
# ============================================================
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import torch
from itertools import islice

@dataclass
class CausalCollator:
    pad_token_id: int
    def __call__(self, batch):
        ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
        ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
        am = [torch.ones_like(x) for x in ii]
        ii = pad_sequence(ii, batch_first=True, padding_value=self.pad_token_id)
        ll = pad_sequence(ll, batch_first=True, padding_value=-100)
        am = pad_sequence(am, batch_first=True, padding_value=0)
        return {"input_ids": ii, "labels": ll, "attention_mask": am}

collator = CausalCollator(pad_token_id=tokenizer.pad_token_id)
dl = DataLoader(
    tok["train"],
    batch_size=1,          # seguro para T4
    shuffle=True,
    collate_fn=collator,
    num_workers=2,         # evita problemas en Colab
    pin_memory=True
)

batch = next(iter(dl))
for k,v in batch.items():
    print("[7]", k, v.shape, v.dtype)

model.train()
scal_dtype = torch.bfloat16 if bf16_ok else torch.float16

# ------- sanity FORWARD -------
with torch.autocast(device_type="cuda", dtype=scal_dtype):
    out = model(**{k: v.to(model.device, non_blocking=True) for k,v in batch.items()})
print


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is

[7] input_ids torch.Size([1, 2048]) torch.int64
[7] labels torch.Size([1, 2048]) torch.int64
[7] attention_mask torch.Size([1, 2048]) torch.int64


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [ ]:
# ============================================================
# 8) sanity backward (un paso manual)
# ============================================================
import torch

# Optimizador solo sobre los parámetros entrenables (LoRA)
optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4
)

# Tomamos un batch aleatorio
batch = next(iter(dl))
batch = {k: v.to(model.device, non_blocking=True) for k, v in batch.items()}

optim.zero_grad(set_to_none=True)

# Forward + backward en autocast (FP16 para T4)
dtype = torch.bfloat16 if bf16_ok else torch.float16
with torch.autocast(device_type="cuda", dtype=dtype):
    out = model(**batch)
loss = out.loss
print(f"[8] loss: {float(loss):.4f} | requires_grad: {loss.requires_grad}")

# Backward + step
loss.backward()
optim.step()

print("[8] backward y step OK ✅")

# (Opcional) verificar gradientes en LoRA
grad_tensors = [n for n,p in model.named_parameters() if p.requires_grad and p.grad is not None]
print(f"[8] Gradientes activos en {len(grad_tensors)} módulos LoRA (OK si >0)")


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is

[8] loss: 1.3283 | requires_grad: True
[8] backward y step OK ✅
[8] Gradientes activos en 64 módulos LoRA (OK si >0)


In [ ]:
# ============================================================
# 9) Trainer — validación del pipeline (1 época)
# ============================================================
from transformers import Trainer, TrainingArguments
import math, torch

EPOCHS = 1
BATCH_SIZE = 1
ACCUM_STEPS = 16
WARMUP_RATIO = 0.1
SEED = 42

n_train = len(tok["train"])
steps_per_epoch = max(1, math.ceil(n_train / (BATCH_SIZE * ACCUM_STEPS)))
warmup_steps = int(WARMUP_RATIO * steps_per_epoch * EPOCHS)

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/MAIA-PROYECTO/outputs/phi3.5-mini-qlora",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=30,
    do_eval=True,
    eval_steps=30,
    save_steps=30,
    save_total_limit=3,
    bf16=bf16_ok,                  # False en T4 → usa FP16 automáticamente
    fp16=not bf16_ok,
    gradient_checkpointing=True,   # mantiene memoria baja
    dataloader_pin_memory=True,
    dataloader_num_workers=0,      # evita conflictos en Colab
    seed=SEED, data_seed=SEED,
    report_to=None,                # sin W&B ni TB
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    data_collator=collator,
)

print(f"[9] Entrenando… pasos/época = {steps_per_epoch}")
trainer.train()
print("[9] Fin de trainer.train() ✅")


[9] Entrenando… pasos/época = 245


wandb: Currently logged in as: jsoa9012 (jsoa9012-universidad-de-los-andes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]


Step,Training Loss
30,1.387700
60,1.250600
90,1.235200
120,1.243000
150,1.220200
180,1.215900
210,1.221500
240,1.212600


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is

[9] Fin de trainer.train() ✅


In [ ]:
#  10) Guardar modelo
# Guarda los pesos finales del modelo ajustado
trainer.save_model("/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora")

# Guarda el tokenizer para inferencia
tokenizer.save_pretrained("/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora")

('/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/tokenizer_config.json',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/special_tokens_map.json',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/chat_template.jinja',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/tokenizer.model',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/added_tokens.json',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/tokenizer.json')

In [ ]:
# # 10) Evaluar modelo
trainer.evaluate()

/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]


{'eval_loss': 1.1961791515350342,
 'eval_runtime': 580.344,
 'eval_samples_per_second': 0.837,
 'eval_steps_per_second': 0.837,
 'epoch': 1.0}

In [2]:
# ============================================================
# Generar PLS con Phi-3.5-mini-instruct (QLoRA o bf16)
# Columnas esperadas: name, article, summary
# Entorno probado: transformers==4.42.4, accelerate==0.33.0, peft==0.9.0
# ============================================================
import re
import json, shutil, tempfile
import os
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel  # peft==0.9.0 recomendado

# --------- Parámetros generales ----------
DATA_DIR   = Path("/content/drive/MyDrive/MAIA-PROYECTO")
RESULTS_DIR= Path("/content/drive/MyDrive/MAIA-PROYECTO/models/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "microsoft/phi-3.5-mini-instruct"
LORA_DIR   = "/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora"  # ajusta si cambia
CKPT_DIR   = "/content/drive/MyDrive/MAIA-PROYECTO/outputs/phi3.5-mini-qlora/checkpoint-240"  # opcional


# Usa 4-bit si hay bitsandbytes con CUDA; si falla, pon False y carga en bf16
USE_4BIT   = True

# --------- Carga de datos ----------
df = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
for col in ["name", "article", "summary"]:
    assert col in df.columns, f"Falta la columna requerida: {col}"

# --------- Prompt coherente con tu entrenamiento ----------
def prompt_cot_factual(t):
    return f"""You are a helpful medical writer.
            Think briefly before answering:
            - Use only statements explicitly present in the source.
            - Keep names and numbers exactly as written.
            - 4–6 sentences, ≤120 words. Do not show your reasoning.

            Scientific text:
            {t}

            Plain summary:"""
# --------- Carga tokenizer ----------
print("[Modelo] Cargando tokenizer base…")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"   # más robusto en lotes para causal LM

# --------- Carga modelo base (4-bit o bf16) ----------
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
torch.backends.cuda.matmul.allow_tf32 = True

if USE_4BIT:
    try:
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16
        )
        print("[Modelo] Cargando modelo base en 4-bit…")
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            device_map="auto",
            quantization_config=bnb_cfg,
            torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
            attn_implementation="eager",   # evita flash-attn/triton
            trust_remote_code=True
        )
    except Exception as e:
        print(f"⚠️ 4-bit no disponible ({e}). Cargando en bf16…")
        USE_4BIT = False

if not USE_4BIT:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="auto",
        torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
        attn_implementation="eager",
        trust_remote_code=True
    )

# --------- Montar adaptador LoRA ----------
print("[Modelo] Montando adaptador LoRA fine-tuneado…")

def sanitize_lora_config(src_dir: Path) -> Path:
    """Crea una copia temporal del adaptador LoRA quitando claves no soportadas (p. ej. use_qalora)."""
    cfg_path = src_dir / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No se encontró {cfg_path}")
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    # Elimina claves nuevas que peft 0.9.0 no reconoce
    drop_keys = ["use_qalora", "rank_dropout", "module_dropout", "alpha_pattern",
                 "init_lora_weights", "fan_in_fan_out", "quant_storage_dtype"]
    for k in drop_keys:
        cfg.pop(k, None)

    tmp_dir = Path(tempfile.mkdtemp(prefix="lora_sanitized_"))
    shutil.copytree(src_dir, tmp_dir, dirs_exist_ok=True)
    with open(tmp_dir / "adapter_config.json", "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)
    return tmp_dir

try:
    sanitized_lora = sanitize_lora_config(Path(LORA_DIR))
    model = PeftModel.from_pretrained(base_model, str(sanitized_lora))
    if CKPT_DIR and os.path.isdir(CKPT_DIR):
        try:
            model.load_adapter(CKPT_DIR, adapter_name="resume")
            model.set_adapter("resume")
        except Exception as e:
            print("ℹ️ No se aplicó checkpoint adicional:", e)
    print("✅ Adaptador LoRA cargado correctamente.")
except Exception as e:
    print(f"❌ Error cargando LoRA ({e}) → usando solo modelo base.")
    model = base_model

model.eval()
print(f"✅ Modelo listo en {next(model.parameters()).device}, dtype={next(model.parameters()).dtype}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

[Modelo] Cargando tokenizer base…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

[Modelo] Montando adaptador LoRA fine-tuneado…
✅ Adaptador LoRA cargado correctamente.
✅ Modelo listo en cuda:0, dtype=torch.bfloat16
PyTorch: 2.8.0+cu126 | device: cuda


In [ ]:
# ---- 6) Inferencia por lotes ----
articulos = df["article"].fillna("").astype(str).tolist()
pls = []

print(f"[Inferencia] Generando {len(articulos)} resúmenes…")
for i in tqdm(range(0, len(articulos), BATCH_SIZE)):
    batch = articulos[i:i+BATCH_SIZE]
    prompts = [generar_prompt(t) for t in batch]
    pls.extend(generate_batch(prompts))

# ---- 7) Guardar resultados ----
df["pls_phi35"] = pls
csv_out = RESULTS_DIR / "summaries_phi35.csv"
df.to_csv(csv_out, index=False, encoding="utf-8")
print(f"✅ Guardado CSV con resúmenes en: {csv_out}")

[Inferencia] Generando 380 resúmenes…


  0%|          | 0/190 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


✅ Guardado CSV con resúmenes en: /content/drive/MyDrive/MAIA-PROYECTO/models/results/summaries_phi35.csv


In [8]:
# ============================================================
# Eval de pérdidas de validación por checkpoint
# - Carga cada checkpoint-XX con PeftModel.from_pretrained(...)
# - Enmascara el prompt (-100) para que la loss mida SOLO el resumen
# - Escribe CSV: step, avg_token_loss, perplexity, valid_tokens
# ============================================================
import os, csv, math, gc, torch
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM
from peft import PeftModel

# ---------- Config ----------
# Si no existe USE_4BIT en tu notebook, define un default:
try:
    USE_4BIT
except NameError:
    USE_4BIT = False

# Carpeta base de checkpoints (padre de checkpoint-240)
CKPT_BASE_DIR = Path("/content/drive/MyDrive/MAIA-PROYECTO/outputs") # ej: .../outputs/phi3.5-mini-qlora
STEPS = [30, 60, 90, 120, 150, 180, 210, 240]
OUT_CSV = RESULTS_DIR / "phi35_eval_por_step_restored.csv"

# ---------- Dataset de validación (prompt enmascarado) ----------
MAX_LEN = 1024
tokenizer.padding_side = "left"

def build_pair(article: str, summary: str):
    prompt = prompt_cot_factual(str(article)).rstrip() + "\n"
    target = str(summary).strip()

    enc_all = tokenizer(prompt + target, max_length=MAX_LEN, truncation=True, return_tensors="pt")
    enc_p   = tokenizer(prompt, max_length=MAX_LEN, truncation=True, return_tensors="pt")

    input_ids = enc_all["input_ids"][0]
    attn      = enc_all["attention_mask"][0]
    labels    = input_ids.clone()

    prompt_len = enc_p["input_ids"].size(1)
    labels[:prompt_len] = -100  # ignora prompt en la pérdida

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

eval_records = [build_pair(a, s) for a, s in zip(df["article"], df["summary"])]

def collate_fn(features):
    keys = features[0].keys()
    batch = {}
    for k in keys:
        tensors = [f[k] for f in features]
        pad_val = tokenizer.pad_token_id if k != "labels" else -100
        batch[k] = torch.nn.utils.rnn.pad_sequence(tensors, batch_first=True, padding_value=pad_val)
    return batch

eval_loader = DataLoader(
    eval_records, batch_size=4, shuffle=False, num_workers=0,
    collate_fn=collate_fn, pin_memory=torch.cuda.is_available()
)

# ---------- Carga base model (4-bit o bf16/fp16) ----------
def make_base_model():
    bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    if USE_4BIT:
        try:
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16
            )
            return AutoModelForCausalLM.from_pretrained(
                BASE_MODEL,
                device_map="auto",
                quantization_config=bnb_cfg,
                torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
                attn_implementation="eager",
                trust_remote_code=True
            )
        except Exception as e:
            print(f"⚠️ 4-bit no disponible ({e}). Intentando bf16/fp16…")
    # fallback bf16/fp16
    return AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="auto",
        torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
        attn_implementation="eager",
        trust_remote_code=True
    )

# ---------- Evaluación (avg token loss & ppl) ----------
def evaluate_model(cur_model):
    cur_model.eval()
    device = next(cur_model.parameters()).device
    use_amp = (device.type == "cuda")
    scaler_dtype = torch.bfloat16 if (use_amp and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            if use_amp:
                ctx = torch.amp.autocast("cuda", dtype=scaler_dtype)  # API nueva
            else:
                ctx = torch.no_grad()
            with ctx:
                out = cur_model(**batch, use_cache=False)   # ← clave
                logits = out.logits
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = batch["labels"][:, 1:].contiguous()
                ce = torch.nn.functional.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                    ignore_index=-100,
                    reduction="none"
                ).view(shift_labels.size())
                valid = (shift_labels != -100)
                total_loss += float((ce * valid).sum().item())
                total_tokens += int(valid.sum().item())

    avg_token_loss = total_loss / max(1, total_tokens)
    try:
        ppl = math.exp(avg_token_loss)
    except OverflowError:
        ppl = float("inf")
    return avg_token_loss, ppl, total_tokens

# ---------- Loop por checkpoints ----------
if not OUT_CSV.exists():
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["step", "avg_token_loss", "perplexity", "valid_tokens"])

for step in STEPS:
    ckpt_path = CKPT_BASE_DIR / f"checkpoint-{step}"
    if not ckpt_path.exists():
        print(f"⚠️ Omitido: no existe {ckpt_path}")
        continue

    print(f"[Eval] Cargando checkpoint {ckpt_path} ...")
    base_tmp = make_base_model()

    model_tmp = PeftModel.from_pretrained(base_tmp, str(ckpt_path))
    model_tmp = PeftModel.from_pretrained(base_tmp, str(ckpt_path))
    model_tmp.config.use_cache = False          # ← desactiva KV cache
    avg_loss, ppl, ntok = evaluate_model(model_tmp)
    print(f" step={step}  loss={avg_loss:.6f}  ppl={ppl:.3f}  tokens={ntok}")

    with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([step, avg_loss, ppl, ntok])

    # Limpieza
    del model_tmp, base_tmp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✅ Resultados guardados en:", OUT_CSV)

[Eval] Cargando checkpoint /content/drive/MyDrive/MAIA-PROYECTO/outputs/checkpoint-30 ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


 step=30  loss=1.353171  ppl=3.870  tokens=14813
[Eval] Cargando checkpoint /content/drive/MyDrive/MAIA-PROYECTO/outputs/checkpoint-60 ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

 step=180  loss=1.279660  ppl=3.595  tokens=14813
[Eval] Cargando checkpoint /content/drive/MyDrive/MAIA-PROYECTO/outputs/checkpoint-210 ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 step=210  loss=1.278310  ppl=3.591  tokens=14813
[Eval] Cargando checkpoint /content/drive/MyDrive/MAIA-PROYECTO/outputs/checkpoint-240 ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 step=240  loss=1.278383  ppl=3.591  tokens=14813
✅ Resultados guardados en: /content/drive/MyDrive/MAIA-PROYECTO/models/results/phi35_eval_por_step_restored.csv
